# P05 — División en entrenamiento, validación y prueba
## (a) Paso del protocolo y objetivo
**Paso P05**: repartir el corpus en **entrenamiento / validación / prueba** con la semilla 42, asignando **grupos completos** de paráfrasis (P04). En los lotes reales: el **lote 1** aporta validación y prueba reales (entrenamiento sintético); el **lote 2** va **solo a prueba** (prueba independiente de G3) y su entrenamiento es de 943 frases. Sustenta el **OE2** y la compuerta **G3**.
**Origen:** partición **Sintética** en vivo; particiones **Reales** como conteos (recalculados con el paquete privado, si está).

## (b) Código del repositorio que lo implementa
`split_corpus.py` (70/15/15 por grupos), `split_corpus_v3.py` (lote 1 real), `preparar_entrenamiento_lote2.py` (arma las 943 frases sin usar ninguna del lote 2) y `split_lote2.py` (solo test, exclusión de idénticas al entrenamiento). Entradas: `corpus/corpus_metadata.csv`, frases reales validadas. Salidas: `corpus/dataset_split.csv`, `corpus/v3_real/`, `corpus/v3_lote2/` (estas dos son privadas).

In [1]:
# ----- Preparación mínima (igual en todos los cuadernos) -----
import os, sys, zipfile
from pathlib import Path

EN_COLAB = "google.colab" in sys.modules
if EN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    CARPETA_DRIVE = Path("/content/drive/MyDrive/MPSR_colab")
    BASE = Path("/content/mpsr")
else:
    CARPETA_DRIVE = Path(os.environ.get("MPSR_DRIVE", ".")).resolve()
    BASE = Path(os.environ.get("MPSR_BASE", "mpsr_trabajo")).resolve()
BASE.mkdir(parents=True, exist_ok=True)
ZIP_PUBLICO = CARPETA_DRIVE / "MPSR_colab_publico.zip"
if not ZIP_PUBLICO.exists():
    raise SystemExit(f"Falta {ZIP_PUBLICO}. Sube MPSR_colab_publico.zip a la carpeta MPSR_colab de tu Drive (ver LEEME_colab_por_paso.md).")
if not (BASE / "scripts" / "colab_util.py").exists():          # solo descomprime la primera vez
    with zipfile.ZipFile(ZIP_PUBLICO) as z:
        z.extractall(BASE)
sys.path.insert(0, str(BASE / "scripts"))
import colab_util as U
U.iniciar(BASE, CARPETA_DRIVE)                                  # rutas, paquete privado (si existe) y detección de Rasa
from colab_util import *                                        # rotulo, comparar, sha256, leer_json, tabla_scripts, ic_bootstrap, ...

Listo · BASE = base · paquete privado: SÍ · Rasa: no instalado (no hace falta)


In [2]:
tabla_scripts(scripts_de("05"))

▌ORIGEN: Código del proyecto (no es dato del estudio)


,script,paso del protocolo,propósito (docstring),entradas,salidas,cómo ejecutarlo,usa Rasa,rol en este cuaderno
0,preparar_entrenamiento_lote2.py,Lote 2 (V1.6 5.8),"Lote 2 (prueba independiente de G3) — conjunto de ENTRENAMIENTO, armado SIN usar ninguna frase del lote 2.",707 sintéticas + 185 reales activas + 51 sintéticas nuevas,corpus/v3_lote2/entrenamiento_lote2.csv (943 frases),python scripts/preparar_entrenamiento_lote2.py --solo-contar,No,Documentado (datos reales)
1,split_corpus.py,P05,"P05 — Partición 70/15/15 agrupada por base_phrase_id (sin fuga, P04).",corpus_metadata.csv,corpus/dataset_split.csv (train/validation/test agrupados por base_phrase_id),python scripts/split_corpus.py,No,Lógica recalculada (etapa 4)
2,split_corpus_v3.py,"P05, T05 (lote 1)","Partición v3 (protocolo V1.2, P05/T05): entrenamiento sintético, validación y test con lenguaje real.",Corpus sintético + lote 1 real validado,"corpus/v3_real/ (train sintético; validación y test reales), log de cambios",python scripts/split_corpus_v3.py,No,Documentado (datos reales)
3,split_lote2.py,Lote 2 (P05),Lote 2 (prueba independiente de G3) — partición: las frases del lote 2 van SOLO a test (no hay partición de validación).,Revisión aplicada + entrenamiento + congelamiento previo,corpus/v3_lote2/ (solo test; 13 frases idénticas al entrenamiento excluidas por coincidencia exacta),python scripts/split_lote2.py --help,No,Lógica recalculada (etapa 4)


## (c) Celdas de código
### 1. Partición sintética oficial (Sintético) y su cálculo con la misma lógica de `split_corpus.py`
`split_corpus.py` baraja los grupos de cada intención con `random.Random(42)` y asigna cada grupo a la partición con mayor déficit relativo. Aquí se aplica esa misma función **en memoria** y se compara con la partición guardada; la oficial de la v3 conserva la validación y el test de la v2 (todos los grupos nuevos de v3 están en `train`), por lo que la comparación es **informativa**, no una igualdad exigida.

In [3]:
sys.path.insert(0, str(BASE / "scripts"))
import random
import split_corpus as S                       # solo se usa assign_groups; no se ejecuta main()

cm = pd.read_csv(BASE / "corpus/corpus_metadata.csv", dtype=str, keep_default_na=False, encoding="utf-8")
sumv3 = leer_json("corpus/corpus_summary.json")
rotulo(ORIGEN_SINTETICO, "partición oficial v3 guardada")
display(cm.groupby("split").size().rename("frases").to_frame().T)
for k, v in sumv3["split_counts"].items():
    comparar("P05", f"partición sintética {k}", int((cm["split"] == k).sum()), v, "corpus/corpus_summary.json", tol=0)
comparar("P05", "semilla de la partición", int(pd.read_csv(BASE / "corpus/dataset_split.csv", dtype=str)["seed"].iloc[0]), 42, "corpus/dataset_split.csv", tol=0)

rng = random.Random(42)
asignacion = {}
for intent in sorted(cm["intent"].unique()):
    tam = cm[cm["intent"] == intent].groupby("base_phrase_id").size()
    asignacion.update(S.assign_groups(list(tam.items()), rng))
demo = cm["base_phrase_id"].map(asignacion)
rotulo(ORIGEN_SINTETICO, "DEMOSTRACIÓN — lo que produciría split_corpus.py (objetivo 70/15/15) sobre la v3; NO es la partición oficial")
display(demo.value_counts().rename("frases").to_frame().T)
print("Porcentajes:", (demo.value_counts(normalize=True) * 100).round(1).to_dict(), "→ con solo 4–7 grupos por intención el reparto real se aleja del objetivo nominal 70/15/15 (cada intención debe tener al menos un grupo en validación y uno en prueba).")
print("Grupos repartidos en dos particiones en la demostración:", int((pd.DataFrame({"g": cm["base_phrase_id"], "s": demo}).groupby("g")["s"].nunique() > 1).sum()))

▌ORIGEN: Sintético — partición oficial v3 guardada


split,test,train,validation
frases,81,546,81


▌ORIGEN: Sintético — DEMOSTRACIÓN — lo que produciría split_corpus.py (objetivo 70/15/15) sobre la v3; NO es la partición oficial


base_phrase_id,train,validation,test
frases,384,162,162


Porcentajes: {'train': 54.2, 'validation': 22.9, 'test': 22.9} → con solo 4–7 grupos por intención el reparto real se aleja del objetivo nominal 70/15/15 (cada intención debe tener al menos un grupo en validación y uno en prueba).
Grupos repartidos en dos particiones en la demostración: 0


### 2. Particiones de los lotes reales (Real, solo conteos) y exclusión de idénticas al entrenamiento

In [4]:
# ----- 4b. Lotes REALES: conteos y controles (sin frases) -----
ag = leer_json("recursos/agregados_reales.json")
rotulo(ORIGEN_REAL, "lote 1 (partición v3) y lote 2 — solo conteos")
p1, p2 = ag["particion_lote1"], ag["particion_lote2"]
print("Lote 1 — partición v3:", p1["split"], "| por origen y partición:", p1["origen_por_split"])
print("Lote 2 — partición:", p2["split"], "| entrenamiento por origen:", p2["entrenamiento_por_origen"])
print("Lote 2 — participantes en el test:", p2["participantes_test"], "| situaciones:", p2["situaciones_test"], "| frases por intención (mín–máx):", p2["test_por_intencion_min_max"])

if HAY_PRIVADO:
    rotulo(ORIGEN_REAL, "recalculado desde privado/particion_sin_texto.csv y privado/hash_textos.csv (sin texto)")
    pt = pd.read_csv(PRIV / "particion_sin_texto.csv", dtype=str, keep_default_na=False, encoding="utf-8")
    ht = pd.read_csv(PRIV / "hash_textos.csv", dtype=str, keep_default_na=False, encoding="utf-8")
    l2 = pt[pt["lote"] == "lote2"]
    train2, test2 = l2[l2["split"] == "train"], l2[l2["split"] == "test"]
    print("Lote 2 recalculado — train:", len(train2), "| test:", len(test2), "| excluidas:", int((l2["split"] == "excluida").sum()))
    print("Entrenamiento por origen:", train2["source"].str.replace(r"\s*\(.*", "", regex=True).value_counts().to_dict(), "→ detalle:", train2["source"].value_counts().to_dict())
    h2 = ht[ht["lote"] == "lote2"]
    solape_texto = len(set(h2[h2["split"] == "train"]["sha256"]) & set(h2[h2["split"] == "test"]["sha256"]))
    part_train = set(train2.loc[train2["participant_code"] != "", "participant_code"])
    part_test = set(test2["participant_code"])
    solape_part = len(part_train & part_test)
    l1 = pt[pt["lote"] == "lote1"]
    gp = l1[l1["split"].isin(["train", "validation", "test"])].groupby("base_phrase_id")["split"].nunique()
    solape_grupos = int((gp > 1).sum())
    print("Solapamiento de TEXTO EXACTO entre entrenamiento y test del lote 2 (hash del texto normalizado):", solape_texto)
    print("Participantes presentes a la vez en el entrenamiento real y en el test del lote 2:", solape_part)
    print("Grupos de paráfrasis del lote 1 en más de una partición:", solape_grupos)
    comparar("P04/P05", "lote 2: frases de test", len(test2), 267, "Protocolo V1.8 / particion_lote2_cifras.md", tol=0)
    comparar("P04/P05", "lote 2: frases de entrenamiento", len(train2), 943, "Protocolo V1.8 (707 + 185 + 51)", tol=0)
    comparar("P04/P05", "lote 2: frases excluidas por idénticas al entrenamiento", int((l2["split"] == "excluida").sum()), 13, "Protocolo V1.8 (13 de 280)", tol=0)
    comparar("P04/P05", "lote 2: solapamientos de texto exacto test/entrenamiento", solape_texto, 0, "Protocolo V1.8 («0 solapamientos»)", tol=0)
    comparar("P04/P05", "lote 2: participantes compartidos entrenamiento/test", solape_part, 0, "diseño del lote 2 (solo test, otros participantes)", tol=0)
    ex = pd.read_csv(PRIV / "exclusion_entrenamiento_lote2.csv", dtype=str, keep_default_na=False, encoding="utf-8")
    print("Exclusión por coincidencia exacta:", len(ex), "de 280 recibidas | por fuente:", ex["fuente"].value_counts().to_dict())
else:
    requiere_privado("recálculo de las particiones reales y de los controles de solapamiento")
    c = ag["controles"]
    print("Valores guardados:", c)
    comparar("P04/P05", "lote 2: frases de test (guardado)", p2["split"]["test"], 267, "Protocolo V1.8", tol=0)
    comparar("P04/P05", "lote 2: frases de entrenamiento (guardado)", p2["split"]["train"], 943, "Protocolo V1.8", tol=0)
    comparar("P04/P05", "lote 2: excluidas por idénticas al entrenamiento (guardado)", p2["split"]["excluida"], 13, "Protocolo V1.8", tol=0)
    comparar("P04/P05", "lote 2: solapamiento de texto exacto test/entrenamiento (guardado)", c["lote2_solape_texto_exacto_test_vs_entrenamiento"], 0, "Protocolo V1.8", tol=0)

▌ORIGEN: Real — lote 1 (partición v3) y lote 2 — solo conteos
Lote 1 — partición v3: {'train': 707, 'test': 114, 'validation': 71, 'excluida': 2} | por origen y partición: {'excluida': {'sintético': 1, 'lenguaje real (lote 1)': 1}, 'test': {'lenguaje real (lote 1)': 114}, 'train': {'sintético': 707}, 'validation': {'lenguaje real (lote 1)': 71}}
Lote 2 — partición: {'train': 943, 'test': 267, 'excluida': 13} | entrenamiento por origen: {'sintético': 707, 'lenguaje real (lote 1)': 185, 'sintético (refinamiento ciclo 1)': 51}
Lote 2 — participantes en el test: 25 | situaciones: 56 | frases por intención (mín–máx): [4, 15]
▌ORIGEN: Real — recalculado desde privado/particion_sin_texto.csv y privado/hash_textos.csv (sin texto)
Lote 2 recalculado — train: 943 | test: 267 | excluidas: 13
Entrenamiento por origen: {'sintético': 758, 'lenguaje real': 185} → detalle: {'sintético': 707, 'lenguaje real (lote 1)': 185, 'sintético (refinamiento ciclo 1)': 51}
Solapamiento de TEXTO EXACTO entre entre

### 3. Cómo se armó el entrenamiento del modelo congelado (Real, solo conteos)

In [5]:
ent = ag["particion_lote2"]["entrenamiento_por_origen"]
rotulo(ORIGEN_REAL, "composición del entrenamiento del modelo congelado (943 frases)")
display(pd.DataFrame({"origen de las frases": list(ent), "frases": list(ent.values())}))
comparar("P05", "entrenamiento = 707 sintéticas + 185 reales del lote 1 + 51 sintéticas nuevas", sum(ent.values()), 943, "Protocolo V1.8", tol=0)
comparar("P05", "sintéticas de partida", sum(v for k, v in ent.items() if k.startswith("sint") and "refinamiento" not in k), 707, "Protocolo V1.8", tol=0)
comparar("P05", "reales del lote 1 activas", sum(v for k, v in ent.items() if "lote 1" in k), 185, "Protocolo V1.8", tol=0)
comparar("P05", "sintéticas nuevas (refinamiento ciclo 1)", sum(v for k, v in ent.items() if "refinamiento" in k), 51, "Protocolo V1.8", tol=0)
ref = pd.read_csv(BASE / "corpus/refinamiento_lote2/sinteticas_ciclo1.csv", dtype=str, keep_default_na=False, encoding="utf-8")
print("Sintéticas del refinamiento (archivo del paquete):", len(ref), "| intenciones:", ref["intent"].nunique())
comparar("P05", "frases en sinteticas_ciclo1.csv", len(ref), 51, "corpus/refinamiento_lote2/sinteticas_ciclo1.csv", tol=0)

▌ORIGEN: Real — composición del entrenamiento del modelo congelado (943 frases)


,origen de las frases,frases
0,sintético,707
1,lenguaje real (lote 1),185
2,sintético (refinamiento ciclo 1),51


Sintéticas del refinamiento (archivo del paquete): 51 | intenciones: 7


In [6]:
cierre("P05 División")

P05 División — recalculado frente a reportado: 14 cifras · coinciden: 14 · NO coinciden: 0


,paso,cifra,recalculado,reportado,fuente de lo reportado,coincide
0,P05,partición sintética train,546,546,corpus/corpus_summary.json,Sí
1,P05,partición sintética validation,81,81,corpus/corpus_summary.json,Sí
2,P05,partición sintética test,81,81,corpus/corpus_summary.json,Sí
3,P05,semilla de la partición,42,42,corpus/dataset_split.csv,Sí
4,P04/P05,lote 2: frases de test,267,267,Protocolo V1.8 / particion_lote2_cifras.md,Sí
5,P04/P05,lote 2: frases de entrenamiento,943,943,Protocolo V1.8 (707 + 185 + 51),Sí
6,P04/P05,lote 2: frases excluidas por idénticas al entrenamiento,13,13,Protocolo V1.8 (13 de 280),Sí
7,P04/P05,lote 2: solapamientos de texto exacto test/entrenamiento,0,0,Protocolo V1.8 («0 solapamientos»),Sí
8,P04/P05,lote 2: participantes compartidos entrenamiento/test,0,0,"diseño del lote 2 (solo test, otros participantes)",Sí
9,P05,entrenamiento = 707 sintéticas + 185 reales del lote 1 + 51 sintéticas nuevas,943,943,Protocolo V1.8,Sí


DISCREPANCIAS (recalculado ≠ reportado): ninguna
Rasa disponible: False · paquete privado: True


## (d) Cómo leer el resultado
- **Sintético:** 546/81/81 con semilla 42 y 0 grupos repartidos. La demostración con `assign_groups` muestra el procedimiento (objetivo nominal 70/15/15; el reparto real depende de los pocos grupos por intención) sin alterar la partición oficial.
- **Lote 1 (real):** entrenamiento sintético 707, validación 71 y prueba 114 (2 excluidas). **Lote 2 (real):** solo prueba — 267 frases de 25 participantes, tras **excluir 13 de 280** por ser idénticas (coincidencia exacta normalizada) a una del entrenamiento (7 sintéticas y 6 reales del lote 1); no hay partición de validación.
- **Entrenamiento del modelo congelado:** 707 sintéticas + 185 reales del lote 1 + 51 sintéticas nuevas = **943**, armado sin usar ninguna frase del lote 2.

## Limitaciones
- Las frases del lote 2 provienen de las mismas 56 situaciones del lote 1; 4–5 frases por intención en el test.
- La exclusión de 13 frases reduce el test de 280 a 267, pero se hace por criterio objetivo declarado antes de medir, no por rendimiento.